# Structure vs Effect — Decision-Level Experiment v1

**Publication scope:** this paper only  
**Protocol:** `configs/DECISION_PROTOCOL_V1.md`  
**Protocol version:** `DECISION_PROTOCOL_V1`  
**Experiment status:** Primary decision-level execution

This notebook implements the frozen protocol without changing its research
questions, oracle, structural-indicator policies, abstention rule, regret definition,
or primary metrics. It reads already-frozen Method D and B2A outputs; it does
not refit, tune, or introduce a model.

All findings remain restricted to the NeurIPS 2022 Causal Education synthetic
local-development benchmark. B2A results remain development-stage evidence
because its estimator was developed after Task 2 ground truth exposure.


## 1. Canonical inputs and provenance

The experiment uses these inputs only:

1. `results/task1_method_d_edges.csv` — frozen learned Method D edges.
2. `results/task2_b2a_effect_estimates.csv` — frozen B2A estimates and query keys.
3. `Task_1_local_public.zip::Task_1_data_local_dev_csv/adj_matrix.npy` —
   evaluation-only ground-truth structures.
4. `Task_2_local_public.zip::Task_2_data_local_dev/intervention_0.json` through
   `intervention_4.json` — canonical query definitions.
5. `Task_2_local_public.zip::Task_2_data_local_dev/cate_estimate.npy` —
   evaluation-only ground-truth effects.

A benchmark alignment diagnostic found zero literal trajectory matches across
the 50 queries. This does not invalidate the mapping: the benchmark defines Task 2
query groups by dataset index, so `Task 2 set_i -> Task 1 dataset_i` is the
intended same-index mapping. Conditioning histories are simulated paths and
need not be literal excerpts of Task 1 training trajectories. The mapping is
therefore benchmark-defined, not trajectory-match-derived. The
structure-versus-effect decision analysis uses `set_id` as `dataset_id`.


In [ ]:
# تهيئ هذه الخلية المكتبات والمسارات الثابتة لضمان تنفيذ قابل لإعادة الإنتاج.
import hashlib
import io
import json
import math
import zipfile
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd

PROTOCOL_VERSION = "DECISION_PROTOCOL_V1"
EXPECTED_QUERY_COUNT = 50
EXPECTED_ENVIRONMENTS = tuple(range(5))
EXPECTED_QUERIES_PER_ENVIRONMENT = 10
BOOTSTRAP_REPS = 10_000
BOOTSTRAP_SEED = 20260908

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROTOCOL_PATH = PROJECT_ROOT / "configs/DECISION_PROTOCOL_V1.md"
METHOD_D_PATH = PROJECT_ROOT / "results/task1_method_d_edges.csv"
B2A_PATH = PROJECT_ROOT / "results/task2_b2a_effect_estimates.csv"
RAW_DATA_DIR = PROJECT_ROOT / "data/neurips-2022-causal-education"
TASK1_ZIP_PATH = RAW_DATA_DIR / "Task_1_local_public.zip"
TASK2_ZIP_PATH = RAW_DATA_DIR / "Task_2_local_public.zip"
TASK1_GT_MEMBER = "Task_1_data_local_dev_csv/adj_matrix.npy"
TASK2_GT_MEMBER = "Task_2_data_local_dev/cate_estimate.npy"
TASK2_QUERY_MEMBERS = [
    f"Task_2_data_local_dev/intervention_{dataset_id}.json"
    for dataset_id in EXPECTED_ENVIRONMENTS
]
OUTPUT_DIR = PROJECT_ROOT / "results"

required_paths = [
    PROTOCOL_PATH,
    METHOD_D_PATH,
    B2A_PATH,
    TASK1_ZIP_PATH,
    TASK2_ZIP_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
assert not missing_paths, f"Missing canonical inputs: {missing_paths}"
print("Repository root:", PROJECT_ROOT)
print("Protocol version:", PROTOCOL_VERSION)


In [ ]:
# تحمّل هذه الخلية تعريفات الاستعلام والتنبؤات المجمدة قبل فتح حقائق التقييم.
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_path(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


input_provenance = [
    {
        "role": "frozen_protocol",
        "container": str(PROTOCOL_PATH),
        "member": "",
        "sha256": sha256_path(PROTOCOL_PATH),
    },
    {
        "role": "learned_structure_method_d",
        "container": str(METHOD_D_PATH),
        "member": "",
        "sha256": sha256_path(METHOD_D_PATH),
    },
    {
        "role": "estimated_effect_b2a",
        "container": str(B2A_PATH),
        "member": "",
        "sha256": sha256_path(B2A_PATH),
    },
]

method_d_edges = pd.read_csv(METHOD_D_PATH)
b2a_raw = pd.read_csv(B2A_PATH)

query_rows = []
with zipfile.ZipFile(TASK2_ZIP_PATH, "r") as archive:
    archive_names = set(archive.namelist())
    assert set(TASK2_QUERY_MEMBERS).issubset(archive_names)
    assert TASK2_GT_MEMBER in archive_names

    for dataset_id, member in zip(EXPECTED_ENVIRONMENTS, TASK2_QUERY_MEMBERS):
        member_bytes = archive.read(member)
        queries = json.loads(member_bytes)
        input_provenance.append({
            "role": "task2_query_definitions",
            "container": str(TASK2_ZIP_PATH),
            "member": member,
            "sha256": sha256_bytes(member_bytes),
        })

        assert len(queries) == EXPECTED_QUERIES_PER_ENVIRONMENT
        for query_id, query in enumerate(queries):
            intervention_array = np.asarray(query["intervention"], dtype=float)
            reference_array = np.asarray(query["reference"], dtype=float)
            effect_mask = np.asarray(query["effect_mask"], dtype=bool)
            intervention_values = intervention_array[~np.isnan(intervention_array)]
            reference_values = reference_array[~np.isnan(reference_array)]
            effect_positions = np.argwhere(effect_mask)

            assert intervention_values.size == 1
            assert reference_values.size == 1
            assert effect_positions.shape == (1, 2)
            effect_time, effect_column = effect_positions[0]

            query_rows.append({
                "dataset_id": dataset_id,
                "query_id": query_id,
                "intervention": int(intervention_values[0]),
                "reference": int(reference_values[0]),
                "target": int(effect_column - 1),
                "effect_time": int(effect_time),
            })

queries_df = pd.DataFrame(query_rows)
b2a = b2a_raw.rename(columns={
    "set_id": "dataset_id",
    "target_construct": "target",
    "cate_hat": "cate_hat_b2a",
})

query_key = ["dataset_id", "query_id"]
query_columns = query_key + ["intervention", "reference", "target", "effect_time"]
assert len(queries_df) == EXPECTED_QUERY_COUNT
assert not queries_df.duplicated(query_key).any()
assert not b2a.duplicated(query_key).any()

decision_df = queries_df.merge(
    b2a[query_columns + ["cate_hat_b2a"]],
    on=query_columns,
    how="left",
    validate="one_to_one",
)
assert decision_df["cate_hat_b2a"].notna().all()
print("Frozen predictions and query definitions loaded:", len(decision_df))


In [ ]:
# توثق هذه الخلية مصادر كل مدخل وتتحقق من اتساق خرائط البيئة والاستعلام.
expected_keys = pd.MultiIndex.from_product(
    [EXPECTED_ENVIRONMENTS, range(EXPECTED_QUERIES_PER_ENVIRONMENT)],
    names=query_key,
)
observed_keys = pd.MultiIndex.from_frame(
    decision_df[query_key].sort_values(query_key).reset_index(drop=True)
)
assert observed_keys.equals(expected_keys)
assert set(decision_df["dataset_id"]) == set(EXPECTED_ENVIRONMENTS)
assert decision_df.groupby("dataset_id").size().eq(
    EXPECTED_QUERIES_PER_ENVIRONMENT
).all()
assert decision_df[["intervention", "reference", "target"]].apply(
    lambda column: column.between(0, 49).all()
).all()
assert (decision_df["intervention"] != decision_df["reference"]).all()
assert np.isfinite(decision_df["cate_hat_b2a"]).all()

expected_method_d_columns = {
    "dataset_id", "source_construct", "target_construct", "selected_by_method_d"
}
assert set(method_d_edges.columns) == expected_method_d_columns
assert method_d_edges["selected_by_method_d"].eq(True).all()
assert not method_d_edges.duplicated(
    ["dataset_id", "source_construct", "target_construct"]
).any()
assert set(method_d_edges["dataset_id"]) == set(EXPECTED_ENVIRONMENTS)

provenance_df = pd.DataFrame(input_provenance)
display(provenance_df)
print("Mapping check: Task 2 set_i -> Task 1 dataset_i (benchmark-defined; not trajectory-match-derived).")


## 2. Evaluation-only ground truth

The following cell opens the two ground-truth members only after the frozen
Method D and B2A prediction artifacts have been loaded. These ground truths are
used solely to construct the oracle policies and evaluate the already-frozen
predictions. No prediction is refitted, reconstructed, filtered, or tuned.


In [ ]:
# تفتح هذه الخلية حقائق البنية والأثر للتقييم فقط بعد تثبيت التنبؤات المدخلة.
with zipfile.ZipFile(TASK1_ZIP_PATH, "r") as archive:
    assert TASK1_GT_MEMBER in archive.namelist()
    task1_gt_bytes = archive.read(TASK1_GT_MEMBER)
    gt_adjacency = np.load(io.BytesIO(task1_gt_bytes), allow_pickle=False)

with zipfile.ZipFile(TASK2_ZIP_PATH, "r") as archive:
    task2_gt_bytes = archive.read(TASK2_GT_MEMBER)
    tau_gt_matrix = np.load(io.BytesIO(task2_gt_bytes), allow_pickle=False)

input_provenance.extend([
    {
        "role": "ground_truth_structure_evaluation_only",
        "container": str(TASK1_ZIP_PATH),
        "member": TASK1_GT_MEMBER,
        "sha256": sha256_bytes(task1_gt_bytes),
    },
    {
        "role": "ground_truth_effect_evaluation_only",
        "container": str(TASK2_ZIP_PATH),
        "member": TASK2_GT_MEMBER,
        "sha256": sha256_bytes(task2_gt_bytes),
    },
])
provenance_df = pd.DataFrame(input_provenance)

assert gt_adjacency.shape == (5, 50, 50)
assert tau_gt_matrix.shape == (5, 10)
assert set(np.unique(gt_adjacency)).issubset({0, 1})
assert np.isfinite(tau_gt_matrix).all()

tau_rows = [
    {
        "dataset_id": dataset_id,
        "query_id": query_id,
        "tau_gt": float(tau_gt_matrix[dataset_id, query_id]),
    }
    for dataset_id in EXPECTED_ENVIRONMENTS
    for query_id in range(EXPECTED_QUERIES_PER_ENVIRONMENT)
]
decision_df = decision_df.merge(
    pd.DataFrame(tau_rows),
    on=query_key,
    how="left",
    validate="one_to_one",
)
assert decision_df["tau_gt"].notna().all()
print("Ground-truth structures:", gt_adjacency.shape)
print("Ground-truth effects:", tau_gt_matrix.shape)


## 3. Frozen decision policies

The next cell applies the protocol literally:

- effect sign defines the oracle and B2A choices;
- exactly one direct structural edge to the target defines a structural choice;
- both or neither structural edges produce `ABSTAIN`;
- non-tied oracle queries are classified as sufficient, misleading, or ambiguous;
- regret is missing for abstentions rather than replaced with zero.


In [ ]:
# تطبق هذه الخلية سياسات القرار والتصنيف والندم كما جُمّدت دون أي كسر تعادل لاحق.
def effect_decision(value, tie_label):
    if value > 0:
        return "I"
    if value < 0:
        return "R"
    return tie_label


def structure_decision(intervention_edge, reference_edge):
    if intervention_edge and not reference_edge:
        return "I"
    if reference_edge and not intervention_edge:
        return "R"
    return "ABSTAIN"


def structure_state(structure_choice, oracle_choice):
    if oracle_choice == "EFFECT_TIE":
        return "EFFECT_TIE"
    if structure_choice == "ABSTAIN":
        return "STRUCTURE_AMBIGUOUS"
    if structure_choice == oracle_choice:
        return "STRUCTURE_SUFFICIENT"
    return "STRUCTURE_MISLEADING"


def selective_regret(choice, oracle_choice, tau_gt):
    if choice == "ABSTAIN" or oracle_choice == "EFFECT_TIE":
        return np.nan
    return 0.0 if choice == oracle_choice else abs(float(tau_gt))


method_d_edge_keys = set(zip(
    method_d_edges["dataset_id"].astype(int),
    method_d_edges["source_construct"].astype(int),
    method_d_edges["target_construct"].astype(int),
))

decision_df["oracle_decision"] = decision_df["tau_gt"].map(
    lambda value: effect_decision(value, "EFFECT_TIE")
)
decision_df["b2a_decision"] = decision_df["cate_hat_b2a"].map(
    lambda value: effect_decision(value, "ABSTAIN")
)

for row_index, row in decision_df.iterrows():
    dataset_id = int(row["dataset_id"])
    intervention = int(row["intervention"])
    reference = int(row["reference"])
    target = int(row["target"])

    decision_df.loc[row_index, "gt_struct_I_to_Y"] = bool(
        gt_adjacency[dataset_id, intervention, target]
    )
    decision_df.loc[row_index, "gt_struct_R_to_Y"] = bool(
        gt_adjacency[dataset_id, reference, target]
    )
    decision_df.loc[row_index, "method_d_I_to_Y"] = (
        dataset_id, intervention, target
    ) in method_d_edge_keys
    decision_df.loc[row_index, "method_d_R_to_Y"] = (
        dataset_id, reference, target
    ) in method_d_edge_keys

for prefix in ["gt_structure", "method_d"]:
    i_edge_column = "gt_struct_I_to_Y" if prefix == "gt_structure" else "method_d_I_to_Y"
    r_edge_column = "gt_struct_R_to_Y" if prefix == "gt_structure" else "method_d_R_to_Y"
    decision_column = f"{prefix}_decision"
    state_column = f"{prefix}_state"
    regret_column = f"{prefix}_regret"

    decision_df[decision_column] = [
        structure_decision(i_edge, r_edge)
        for i_edge, r_edge in zip(decision_df[i_edge_column], decision_df[r_edge_column])
    ]
    decision_df[state_column] = [
        structure_state(choice, oracle)
        for choice, oracle in zip(decision_df[decision_column], decision_df["oracle_decision"])
    ]
    decision_df[regret_column] = [
        selective_regret(choice, oracle, tau)
        for choice, oracle, tau in zip(
            decision_df[decision_column],
            decision_df["oracle_decision"],
            decision_df["tau_gt"],
        )
    ]

decision_df = decision_df.sort_values(query_key).reset_index(drop=True)
display(decision_df.head(10))


In [ ]:
# تتحقق هذه الخلية من سلامة صفوف القرار ومن عدم تحويل الغياب أو الامتناع إلى صفر.
valid_effect_actions = {"I", "R", "EFFECT_TIE"}
valid_policy_actions = {"I", "R", "ABSTAIN"}
valid_states = {
    "STRUCTURE_SUFFICIENT",
    "STRUCTURE_MISLEADING",
    "STRUCTURE_AMBIGUOUS",
    "EFFECT_TIE",
}

assert len(decision_df) == EXPECTED_QUERY_COUNT
assert not decision_df.duplicated(query_key).any()
assert set(decision_df["oracle_decision"]).issubset(valid_effect_actions)
assert set(decision_df["gt_structure_decision"]).issubset(valid_policy_actions)
assert set(decision_df["method_d_decision"]).issubset(valid_policy_actions)
assert set(decision_df["b2a_decision"]).issubset(valid_policy_actions)
assert set(decision_df["gt_structure_state"]).issubset(valid_states)
assert set(decision_df["method_d_state"]).issubset(valid_states)

for prefix in ["gt_structure", "method_d"]:
    abstain = decision_df[f"{prefix}_decision"].eq("ABSTAIN")
    tied = decision_df["oracle_decision"].eq("EFFECT_TIE")
    assert decision_df.loc[abstain | tied, f"{prefix}_regret"].isna().all()
    assert decision_df.loc[~(abstain | tied), f"{prefix}_regret"].notna().all()

assert decision_df[[
    "gt_struct_I_to_Y", "gt_struct_R_to_Y",
    "method_d_I_to_Y", "method_d_R_to_Y",
]].notna().all().all()
assert not decision_df[["tau_gt", "cate_hat_b2a"]].isna().any().any()

# عدم وجود الحافة هنا يعني غيابها في رسم كامل، وليس تعويض دليل مفقود بصفر.
assert set(decision_df["dataset_id"]) == set(EXPECTED_ENVIRONMENTS)
print("All frozen-policy and integrity assertions passed.")


## 4. Metrics and uncertainty

Proportion intervals use two-sided 95% Wilson score intervals. Mean selective
regret uses 10,000 bootstrap replicates with seed `20260908`. For pooled
results, decided queries are resampled with replacement *within each
environment*, preserving the observed number of decided queries contributed by
each environment. Per-environment intervals resample decided queries within
that environment. Abstentions never enter the regret denominator.

No null-hypothesis test is introduced. Where `disagreement_with_b2a` is shown
for the `gt_structure` policy, it is a **descriptive / non-primary comparison**.


In [ ]:
# تعرّف هذه الخلية فواصل ويلسون وبوتستراب طبقيًا لحساب عدم اليقين بشفافية.
def safe_ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan


def wilson_interval(successes, total, confidence=0.95):
    if total == 0:
        return (np.nan, np.nan)
    z = NormalDist().inv_cdf(0.5 + confidence / 2)
    proportion = successes / total
    denominator = 1 + z * z / total
    center = (proportion + z * z / (2 * total)) / denominator
    half_width = z * math.sqrt(
        proportion * (1 - proportion) / total + z * z / (4 * total * total)
    ) / denominator
    return (center - half_width, center + half_width)


def bootstrap_mean_regret(decided_rows, regret_column, seed):
    if decided_rows.empty:
        return (np.nan, np.nan)
    groups = [
        group[regret_column].to_numpy(dtype=float)
        for _, group in decided_rows.groupby("dataset_id", sort=True)
    ]
    assert all(len(values) and np.isfinite(values).all() for values in groups)
    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(BOOTSTRAP_REPS, dtype=float)
    for replicate in range(BOOTSTRAP_REPS):
        sample = np.concatenate([
            rng.choice(values, size=len(values), replace=True)
            for values in groups
        ])
        bootstrap_means[replicate] = sample.mean()
    return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))


def interval_fields(prefix, successes, total):
    low, high = wilson_interval(successes, total)
    return {
        prefix: safe_ratio(successes, total),
        f"{prefix}_ci_low": low,
        f"{prefix}_ci_high": high,
    }


In [ ]:
# تحسب هذه الخلية الملخصات المجمعة لكل سياسة مع المقارنات المجمدة والندم الانتقائي.
def summarize_policy(frame, policy, scope, dataset_id, seed_offset):
    eligible = frame.loc[frame["oracle_decision"] != "EFFECT_TIE"].copy()
    decision_column = f"{policy}_decision" if policy != "b2a" else "b2a_decision"
    decided = eligible.loc[eligible[decision_column] != "ABSTAIN"].copy()
    n_total = len(frame)
    n_eligible = len(eligible)
    n_decided = len(decided)
    n_abstain = n_eligible - n_decided
    n_correct = int((decided[decision_column] == decided["oracle_decision"]).sum())
    n_wrong = n_decided - n_correct

    row = {
        "scope": scope,
        "dataset_id": dataset_id,
        "policy": policy,
        "comparator": "ground_truth_effect_oracle",
        "n_total": n_total,
        "n_eligible": n_eligible,
        "n_decided": n_decided,
        "n_abstain": n_abstain,
        **interval_fields("coverage", n_decided, n_eligible),
        **interval_fields("ambiguity_rate", n_abstain, n_eligible),
        **interval_fields("conditional_decision_accuracy", n_correct, n_decided),
        **interval_fields("decision_disagreement", n_wrong, n_decided),
    }

    if policy in {"gt_structure", "method_d"}:
        misleading = int((eligible[f"{policy}_state"] == "STRUCTURE_MISLEADING").sum())
        row.update(interval_fields("misleading_rate", misleading, n_eligible))
        row.update(interval_fields("conditional_misleading_rate", misleading, n_decided))
        regret_values = decided[f"{policy}_regret"].to_numpy(dtype=float)
        regret_low, regret_high = bootstrap_mean_regret(
            decided, f"{policy}_regret", BOOTSTRAP_SEED + seed_offset
        )
        row.update({
            "mean_selective_regret": float(regret_values.mean()) if n_decided else np.nan,
            "mean_selective_regret_ci_low": regret_low,
            "mean_selective_regret_ci_high": regret_high,
            "median_selective_regret": float(np.median(regret_values)) if n_decided else np.nan,
        })

        joint = eligible.loc[
            eligible[decision_column].ne("ABSTAIN")
            & eligible["b2a_decision"].ne("ABSTAIN")
        ]
        joint_disagreement = int((joint[decision_column] != joint["b2a_decision"]).sum())
        row["n_jointly_decided_with_b2a"] = len(joint)
        row.update(interval_fields(
            "disagreement_with_b2a", joint_disagreement, len(joint)
        ))
        row.update({
            "sign_agreement": np.nan,
            "sign_agreement_ci_low": np.nan,
            "sign_agreement_ci_high": np.nan,
        })
    else:
        sign_matches = int((np.sign(frame["tau_gt"]) == np.sign(frame["cate_hat_b2a"])).sum())
        row.update(interval_fields("sign_agreement", sign_matches, n_total))
        for name in ["misleading_rate", "conditional_misleading_rate"]:
            row.update({name: np.nan, f"{name}_ci_low": np.nan, f"{name}_ci_high": np.nan})
        row.update({
            "mean_selective_regret": np.nan,
            "mean_selective_regret_ci_low": np.nan,
            "mean_selective_regret_ci_high": np.nan,
            "median_selective_regret": np.nan,
            "n_jointly_decided_with_b2a": np.nan,
            "disagreement_with_b2a": np.nan,
            "disagreement_with_b2a_ci_low": np.nan,
            "disagreement_with_b2a_ci_high": np.nan,
        })
    return row


summary_rows = [
    summarize_policy(decision_df, policy, "pooled", "ALL", offset)
    for offset, policy in enumerate(["gt_structure", "method_d", "b2a"])
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
# تحسب هذه الخلية المقاييس نفسها منفصلة لكل بيئة مع الحفاظ على ترتيب ثابت.
environment_rows = []
for dataset_id in EXPECTED_ENVIRONMENTS:
    environment_frame = decision_df.loc[decision_df["dataset_id"] == dataset_id]
    for policy_index, policy in enumerate(["gt_structure", "method_d", "b2a"]):
        environment_rows.append(summarize_policy(
            environment_frame,
            policy,
            "environment",
            dataset_id,
            100 + dataset_id * 10 + policy_index,
        ))

environment_summary_df = pd.DataFrame(environment_rows)
display(environment_summary_df)


## 5. Canonical outputs

The query-level table retains the exact inputs, edge indicators, decisions,
states, and selective regrets required by the frozen protocol. Missing regret
for abstentions remains missing in the CSV. The two summary tables contain
pooled and per-environment estimates with uncertainty intervals.


In [ ]:
# تحفظ هذه الخلية الجداول القانونية الثلاثة فقط دون تعديل أي نتيجة سابقة.
query_output_columns = [
    "dataset_id", "query_id", "intervention", "reference", "target",
    "effect_time", "tau_gt", "oracle_decision",
    "gt_struct_I_to_Y", "gt_struct_R_to_Y", "gt_structure_decision",
    "gt_structure_state", "gt_structure_regret",
    "method_d_I_to_Y", "method_d_R_to_Y", "method_d_decision",
    "method_d_state", "method_d_regret", "cate_hat_b2a", "b2a_decision",
]
query_output = decision_df[query_output_columns].copy()

QUERY_OUTPUT_PATH = OUTPUT_DIR / "structure_effect_decision_v1_query_level.csv"
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "structure_effect_decision_v1_summary.csv"
ENVIRONMENT_OUTPUT_PATH = OUTPUT_DIR / "structure_effect_decision_v1_environment_summary.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
query_output.to_csv(QUERY_OUTPUT_PATH, index=False)
summary_df.to_csv(SUMMARY_OUTPUT_PATH, index=False)
environment_summary_df.to_csv(ENVIRONMENT_OUTPUT_PATH, index=False)

for path in [QUERY_OUTPUT_PATH, SUMMARY_OUTPUT_PATH, ENVIRONMENT_OUTPUT_PATH]:
    assert path.is_file() and path.stat().st_size > 0
    print(f"Saved: {path.relative_to(PROJECT_ROOT)} | sha256={sha256_path(path)}")


In [ ]:
# تعرض هذه الخلية النتائج الأولية الأساسية والتحذير العلمي المطلوب للمراجعة.
primary_columns = [
    "policy", "n_eligible", "n_decided", "n_abstain",
    "coverage", "ambiguity_rate", "conditional_decision_accuracy",
    "misleading_rate", "conditional_misleading_rate",
    "decision_disagreement", "mean_selective_regret",
    "mean_selective_regret_ci_low", "mean_selective_regret_ci_high",
    "median_selective_regret", "sign_agreement",
    "disagreement_with_b2a",
]
display(summary_df[primary_columns])

state_counts = pd.concat({
    "gt_structure": decision_df["gt_structure_state"].value_counts(),
    "method_d": decision_df["method_d_state"].value_counts(),
}, axis=1).fillna(0).astype(int)
display(state_counts)

print(
    "Scientific warning: all results are synthetic-benchmark scoped. "
    "B2A metrics are development-stage evidence, not independent validation."
)


## 6. Interpretation boundary

This notebook reports whether direct structural indicators are sufficient,
ambiguous, or misleading for the benchmark intervention decisions. It does not
show that causal structure is unnecessary, that effect estimation is universally
superior, or that any result generalizes to real educational systems.

The protocol remains unchanged. Exploratory path-based policies, practical-effect
thresholds, alternate abstention costs, and post-hoc tie breaking are not part of
this primary execution.


## 7. Post-hoc reviewer-response robustness analysis

This section was motivated by peer-review simulation after Draft 1 and after
the primary direct-edge experiment was completed. It is a **post-hoc
robustness analysis**, not a preregistered analysis, a prospectively frozen
primary evaluation, or independent validation. The 50 queries, effect oracle,
ground-truth graph, retained Method D graph, and all primary outputs remain
unchanged.

The policy was fixed before examining its results in this run: choose `I` iff
`I` reaches `Y` and `R` does not; choose `R` iff `R` reaches `Y` and `I` does
not; otherwise abstain. A direct edge is a path of length one. No weights,
effect magnitudes, path lengths, heuristics, or semantic information enter the
policy.


In [ ]:
# This cell builds ordinary directed reachability and path decisions because the reviewer questioned direct-edge myopia.
PATH_BOOTSTRAP_SEED = 20260908
PAIRED_BOOTSTRAP_SEED = 20260910
PAIRED_BOOTSTRAP_REPS = 10_000
PRIMARY_OUTPUT_PATHS = [
    OUTPUT_DIR / "structure_effect_decision_v1_query_level.csv",
    OUTPUT_DIR / "structure_effect_decision_v1_summary.csv",
    OUTPUT_DIR / "structure_effect_decision_v1_environment_summary.csv",
]
primary_hashes_before_path_analysis = {path.name: sha256_path(path) for path in PRIMARY_OUTPUT_PATHS}

def directed_reachability(adjacency):
    reach = np.asarray(adjacency, dtype=bool).copy()
    for intermediate in range(reach.shape[0]):
        reach |= reach[:, intermediate, None] & reach[None, intermediate, :]
    return reach

method_d_adjacency = np.zeros_like(gt_adjacency, dtype=bool)
for dataset_id, source, target in method_d_edge_keys:
    method_d_adjacency[dataset_id, source, target] = True

path_reachability = {
    "gt": np.stack([directed_reachability(graph) for graph in gt_adjacency]),
    "method_d": np.stack([directed_reachability(graph) for graph in method_d_adjacency]),
}
path_cycles = {
    source: [bool(np.diag(reach[dataset_id]).any()) for dataset_id in EXPECTED_ENVIRONMENTS]
    for source, reach in path_reachability.items()
}

path_rows = []
for structural_source, reach in path_reachability.items():
    for row in decision_df.itertuples(index=False):
        dataset_id = int(row.dataset_id)
        i_reaches = bool(reach[dataset_id, int(row.intervention), int(row.target)])
        r_reaches = bool(reach[dataset_id, int(row.reference), int(row.target)])
        choice = structure_decision(i_reaches, r_reaches)
        state = structure_state(choice, row.oracle_decision)
        path_rows.append({
            "dataset_id": dataset_id, "query_id": int(row.query_id),
            "intervention": int(row.intervention), "reference": int(row.reference),
            "target": int(row.target), "effect_time": int(row.effect_time),
            "tau_gt": float(row.tau_gt), "oracle_decision": row.oracle_decision,
            "structural_source": structural_source, "I_reaches_Y": i_reaches,
            "R_reaches_Y": r_reaches, "path_decision": choice, "path_state": state,
            "selective_regret": selective_regret(choice, row.oracle_decision, row.tau_gt),
            "cycle_detected": path_cycles[structural_source][dataset_id],
        })

path_query_df = pd.DataFrame(path_rows).sort_values(
    ["structural_source", "dataset_id", "query_id"]
).reset_index(drop=True)
assert len(path_query_df) == 2 * EXPECTED_QUERY_COUNT
assert not path_query_df.duplicated(["structural_source", "dataset_id", "query_id"]).any()
print("Cycle audit by source and environment:", path_cycles)


In [ ]:
# This cell computes matched primary-style path metrics and intervals because robustness must be directly comparable.
def bootstrap_path_mean_regret(decided_rows, seed):
    if decided_rows.empty:
        return (np.nan, np.nan)
    groups = [
        group["selective_regret"].to_numpy(dtype=float)
        for _, group in decided_rows.groupby("dataset_id", sort=True)
    ]
    rng = np.random.default_rng(seed)
    means = np.empty(BOOTSTRAP_REPS, dtype=float)
    for replicate in range(BOOTSTRAP_REPS):
        sample = np.concatenate([rng.choice(values, len(values), replace=True) for values in groups])
        means[replicate] = sample.mean()
    return tuple(np.quantile(means, [0.025, 0.975]))

def summarize_path_policy(frame, scope, dataset_id, seed):
    eligible = frame.loc[frame["oracle_decision"] != "EFFECT_TIE"]
    decided = eligible.loc[eligible["path_decision"] != "ABSTAIN"]
    sufficient = int(eligible["path_state"].eq("STRUCTURE_SUFFICIENT").sum())
    misleading = int(eligible["path_state"].eq("STRUCTURE_MISLEADING").sum())
    ambiguous = int(eligible["path_state"].eq("STRUCTURE_AMBIGUOUS").sum())
    n_eligible, n_decided = len(eligible), len(decided)
    regret_low, regret_high = bootstrap_path_mean_regret(decided, seed)
    regrets = decided["selective_regret"].to_numpy(dtype=float)
    row = {
        "scope": scope, "dataset_id": dataset_id,
        "policy": f"{frame['structural_source'].iloc[0]}_path",
        "comparator": "ground_truth_effect_oracle", "n_total": len(frame),
        "n_eligible": n_eligible, "n_decided": n_decided, "n_abstain": ambiguous,
        "structure_sufficient_count": sufficient,
        "structure_misleading_count": misleading,
        "structure_ambiguous_count": ambiguous,
        **interval_fields("coverage", n_decided, n_eligible),
        **interval_fields("ambiguity_rate", ambiguous, n_eligible),
        **interval_fields("conditional_decision_accuracy", sufficient, n_decided),
        **interval_fields("misleading_rate", misleading, n_eligible),
        **interval_fields("conditional_misleading_rate", misleading, n_decided),
        **interval_fields("oracle_disagreement_among_decided", misleading, n_decided),
        "mean_selective_regret": float(regrets.mean()) if n_decided else np.nan,
        "mean_selective_regret_ci_low": regret_low,
        "mean_selective_regret_ci_high": regret_high,
        "median_selective_regret": float(np.median(regrets)) if n_decided else np.nan,
        "cycle_detected": bool(frame["cycle_detected"].any()),
    }
    assert sufficient + misleading == n_decided
    assert sufficient + misleading + ambiguous == n_eligible
    return row

path_summary_rows, path_environment_rows = [], []
for source_index, source in enumerate(["gt", "method_d"]):
    source_frame = path_query_df.loc[path_query_df["structural_source"] == source]
    path_summary_rows.append(summarize_path_policy(
        source_frame, "pooled", "ALL", PATH_BOOTSTRAP_SEED + source_index
    ))
    for dataset_id in EXPECTED_ENVIRONMENTS:
        path_environment_rows.append(summarize_path_policy(
            source_frame.loc[source_frame["dataset_id"] == dataset_id],
            "environment", dataset_id,
            PATH_BOOTSTRAP_SEED + 100 + dataset_id * 10 + source_index,
        ))
path_summary_df = pd.DataFrame(path_summary_rows)
path_environment_summary_df = pd.DataFrame(path_environment_rows)
display(path_summary_df)
display(path_environment_summary_df)


In [ ]:
# This cell runs the paired stratified bootstrap because GT and Method D share the same query indices.
def paired_policy_frame(policy, source):
    if policy == "DIRECT":
        prefix = "gt_structure" if source == "gt" else "method_d"
        frame = decision_df.sort_values(["dataset_id", "query_id"])
        return pd.DataFrame({
            "dataset_id": frame["dataset_id"].to_numpy(dtype=int),
            "decision": frame[f"{prefix}_decision"].to_numpy(),
            "state": frame[f"{prefix}_state"].to_numpy(),
            "regret": frame[f"{prefix}_regret"].to_numpy(dtype=float),
        })
    frame = path_query_df.loc[path_query_df["structural_source"] == source].sort_values(
        ["dataset_id", "query_id"]
    )
    return pd.DataFrame({
        "dataset_id": frame["dataset_id"].to_numpy(dtype=int),
        "decision": frame["path_decision"].to_numpy(),
        "state": frame["path_state"].to_numpy(),
        "regret": frame["selective_regret"].to_numpy(dtype=float),
    })

def paired_metric_values(frame):
    decided = frame["decision"].ne("ABSTAIN")
    return {
        "coverage": float(decided.mean()),
        "ambiguity_rate": float((~decided).mean()),
        "misleading_rate": float(frame["state"].eq("STRUCTURE_MISLEADING").mean()),
        "mean_selective_regret": float(frame.loc[decided, "regret"].mean()),
    }

paired_rows = []
for policy in ["DIRECT", "PATH"]:
    gt_frame = paired_policy_frame(policy, "gt")
    method_frame = paired_policy_frame(policy, "method_d")
    assert gt_frame["dataset_id"].equals(method_frame["dataset_id"])
    gt_observed = paired_metric_values(gt_frame)
    method_observed = paired_metric_values(method_frame)
    bootstrap_differences = {name: np.full(PAIRED_BOOTSTRAP_REPS, np.nan) for name in gt_observed}
    rng = np.random.default_rng(PAIRED_BOOTSTRAP_SEED)
    groups = [
        np.flatnonzero(gt_frame["dataset_id"].to_numpy() == dataset_id)
        for dataset_id in EXPECTED_ENVIRONMENTS
    ]
    for replicate in range(PAIRED_BOOTSTRAP_REPS):
        sampled = np.concatenate([rng.choice(group, len(group), replace=True) for group in groups])
        gt_values = paired_metric_values(gt_frame.iloc[sampled])
        method_values = paired_metric_values(method_frame.iloc[sampled])
        for metric in bootstrap_differences:
            bootstrap_differences[metric][replicate] = gt_values[metric] - method_values[metric]
    for metric in ["coverage", "ambiguity_rate", "misleading_rate", "mean_selective_regret"]:
        finite = bootstrap_differences[metric][np.isfinite(bootstrap_differences[metric])]
        low, high = np.quantile(finite, [0.025, 0.975]) if len(finite) else (np.nan, np.nan)
        paired_rows.append({
            "policy": policy, "metric": metric,
            "difference_definition": "GT_minus_Method_D",
            "gt_estimate": gt_observed[metric],
            "method_d_estimate": method_observed[metric],
            "delta_gt_minus_method_d": gt_observed[metric] - method_observed[metric],
            "bootstrap_ci_low": low, "bootstrap_ci_high": high,
            "bootstrap_replicates": PAIRED_BOOTSTRAP_REPS,
            "valid_bootstrap_replicates": len(finite),
            "bootstrap_seed": PAIRED_BOOTSTRAP_SEED,
            "resampling": "paired_query_indices_within_environment",
        })
paired_comparison_df = pd.DataFrame(paired_rows)
display(paired_comparison_df)


In [ ]:
# This cell writes only the four authorized robustness artifacts and verifies frozen outputs because provenance is critical.
PATH_QUERY_OUTPUT_PATH = OUTPUT_DIR / "structure_effect_path_robustness_v1_query_level.csv"
PATH_SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "structure_effect_path_robustness_v1_summary.csv"
PATH_ENVIRONMENT_OUTPUT_PATH = OUTPUT_DIR / "structure_effect_path_robustness_v1_environment_summary.csv"
PAIRED_OUTPUT_PATH = OUTPUT_DIR / "structure_effect_paired_comparison_v1.csv"
path_query_df.to_csv(PATH_QUERY_OUTPUT_PATH, index=False)
path_summary_df.to_csv(PATH_SUMMARY_OUTPUT_PATH, index=False)
path_environment_summary_df.to_csv(PATH_ENVIRONMENT_OUTPUT_PATH, index=False)
paired_comparison_df.to_csv(PAIRED_OUTPUT_PATH, index=False)

primary_hashes_after_path_analysis = {path.name: sha256_path(path) for path in PRIMARY_OUTPUT_PATHS}
assert primary_hashes_after_path_analysis == primary_hashes_before_path_analysis
for path in [PATH_QUERY_OUTPUT_PATH, PATH_SUMMARY_OUTPUT_PATH, PATH_ENVIRONMENT_OUTPUT_PATH, PAIRED_OUTPUT_PATH]:
    print(f"Saved: {path.relative_to(PROJECT_ROOT)} | sha256={sha256_path(path)}")
print("Frozen primary outputs unchanged:", primary_hashes_after_path_analysis)

direct_rows = summary_df.loc[summary_df["policy"].isin(["gt_structure", "method_d"])].copy()
direct_rows["structural_source"] = direct_rows["policy"].replace({"gt_structure": "gt"})
comparison_columns = ["coverage", "ambiguity_rate", "conditional_decision_accuracy",
                      "misleading_rate", "mean_selective_regret"]
direct_vs_path = direct_rows.set_index("structural_source")[comparison_columns].add_suffix("_direct").join(
    path_summary_df.assign(structural_source=path_summary_df["policy"].str.replace("_path", "", regex=False))
    .set_index("structural_source")[comparison_columns].add_suffix("_path")
)
for metric in comparison_columns:
    direct_vs_path[f"{metric}_path_minus_direct"] = (
        direct_vs_path[f"{metric}_path"] - direct_vs_path[f"{metric}_direct"]
    )
display(direct_vs_path)
print("Prior retained B1 direct-vs-indirect AUROC: pooled=0.8656588841128738; "
      "datasets 0-4=[0.858650, 0.842383, 0.864375, 0.878278, 0.887496].")
print("Method D predates the decision experiment and is retained here without retuning; "
      "it is neither a state-of-the-art claim nor a complete causal decision system.")


## 8. Consolidated Draft-2 reviewer diagnostics

These are post-hoc reviewer-response diagnostics. All 50 locally available
Task-2 development queries are included: 10 per environment for environments
0–4, with no filtering by ground-truth effect, structural edge presence, or
decision outcome. Query definitions were loaded before either evaluation-only
ground truth.

Selective regret is conditional on deciding: it is zero for an oracle-matching
choice and `abs(tau_GT)` otherwise. Abstentions are missing and excluded from
both the mean and median denominators. An always-abstaining policy therefore
has undefined—not zero—selective regret. The prior paired mean-regret difference
uses the same resampled query indices for GT and Method D but averages each side
over its own decided subset. Because those subsets may differ, that comparison
is descriptive and not strictly like-for-like.


In [ ]:
# This cell audits query provenance and direct-to-path transitions because reviewers requested selection transparency.
assert len(decision_df) == 50
assert decision_df.groupby("dataset_id").size().eq(10).all()
assert decision_df[["dataset_id", "query_id"]].drop_duplicates().shape[0] == 50
review_state_order = ["SUFFICIENT", "MISLEADING", "AMBIGUOUS"]
review_transition_rows = []
for source, direct_prefix in [("GT", "gt_structure"), ("METHOD_D", "method_d")]:
    path_source = "gt" if source == "GT" else "method_d"
    direct_states = decision_df.sort_values(query_key)[f"{direct_prefix}_state"].str.replace("STRUCTURE_", "", regex=False)
    path_states = path_query_df.loc[path_query_df["structural_source"] == path_source].sort_values(query_key)["path_state"].str.replace("STRUCTURE_", "", regex=False)
    transition = pd.crosstab(direct_states.to_numpy(), path_states.to_numpy()).reindex(
        index=review_state_order, columns=review_state_order, fill_value=0
    )
    for direct_state in review_state_order:
        for path_state in review_state_order:
            review_transition_rows.append({"structural_source": source, "direct_state": direct_state,
                                           "path_state": path_state, "count": int(transition.loc[direct_state, path_state])})
review_transitions = pd.DataFrame(review_transition_rows)
canonical_transitions = pd.read_csv(OUTPUT_DIR / "draft2_reviewer_structural_transitions_v1.csv")
pd.testing.assert_frame_equal(
    canonical_transitions.loc[canonical_transitions["record_type"] == "transition", review_transitions.columns].reset_index(drop=True),
    review_transitions, check_dtype=False
)
display(canonical_transitions)


In [ ]:
# This cell validates chance and environment-stratified bootstrap outputs because within-environment clustering affects uncertainty.
review_bootstrap = pd.read_csv(OUTPUT_DIR / "draft2_reviewer_structural_bootstrap_v1.csv")
review_chance = pd.read_csv(OUTPUT_DIR / "draft2_reviewer_chance_reference_v1.csv")
assert len(review_bootstrap) == 4 * 7
assert review_bootstrap["bootstrap_replicates"].eq(10_000).all()
assert review_bootstrap["bootstrap_seed"].eq(20260910).all()
assert review_bootstrap["resampling"].eq("query_indices_within_environment_preserving_environment_sizes").all()
assert review_chance["chance_accuracy"].eq(0.50).all()
assert review_chance["p_value_role"].eq("descriptive_only_not_central").all()
display(review_chance)
display(review_bootstrap)


In [ ]:
# This cell verifies Method D audit and sensitivity artifacts because the retained top-k rule must remain identifiable and untuned.
method_d_audit = pd.read_csv(OUTPUT_DIR / "draft2_reviewer_method_d_audit_v1.csv")
method_d_sensitivity = pd.read_csv(OUTPUT_DIR / "draft2_reviewer_method_d_threshold_sensitivity_v1.csv")
assert method_d_audit["frozen_k"].tolist() == [75, 150, 200, 150, 200]
assert method_d_audit["learned_edge_count"].sum() == len(method_d_edges) == 775
assert method_d_audit["rule_frozen_before_decision_experiment"].all()
assert method_d_audit["continuous_score_reconstruction"].eq("exact_reconstruction_validated_against_all_775_canonical_edges").all()
assert method_d_sensitivity["setting_factor"].drop_duplicates().tolist() == [0.75, 0.90, 1.00, 1.10, 1.25]
assert method_d_sensitivity.loc[method_d_sensitivity["setting_factor"] != 1.0, "setting_role"].eq("POST_HOC_SENSITIVITY").all()
display(method_d_audit)
display(method_d_sensitivity)


In [ ]:
# This cell rechecks frozen structural hashes because reviewer diagnostics must not mutate primary evidence.
FROZEN_STRUCTURAL_HASHES_DRAFT2 = {
    "structure_effect_decision_v1_query_level.csv": "d7194b3c53c210dd84d1e70b21a66fb495a306d40a32ecb8ffb00aa4fff7ada4",
    "structure_effect_decision_v1_summary.csv": "bb6b0212f43e80f9a00f7255531ca5c20af0eb75689be930460c720c14bfbbb4",
    "structure_effect_decision_v1_environment_summary.csv": "57bc5f45d4eb86b8e81899fd0ed9132247513113cce4fdedc138154d09f6717d",
    "structure_effect_path_robustness_v1_query_level.csv": "436be6a306464f27aa879bb3aa15aa6083dc6e7b3bf60f1cb5433d2585993272",
    "structure_effect_path_robustness_v1_summary.csv": "51ec1e37ca20eaf20c69f9753e45d895d53d21727cfc9237b52c86ba435769e5",
    "structure_effect_path_robustness_v1_environment_summary.csv": "8bfb461e44d18efb951d6256ac69bf448dee459097e5139790af8bab980a697d",
    "structure_effect_paired_comparison_v1.csv": "8a0b04fda6a164d4414a442952736631a7d4f1b72da5a197211ddfd83f1b915b",
}
for filename, expected_hash in FROZEN_STRUCTURAL_HASHES_DRAFT2.items():
    assert sha256_path(OUTPUT_DIR / filename) == expected_hash
print("All frozen structural outputs remain byte-identical.")
